In [1]:
import requests
import pandas as pd
import sys
if sys.version_info[0] < 3: 
    from StringIO import StringIO
else:
    from io import StringIO
import datetime
import re
import unicodedata
import json


def no_accent_vietnamese(s):
    s = re.sub(u'Đ', 'D', s)
    s = re.sub(u'đ', 'd', s)
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()


response = requests.get('https://vnexpress.net/microservice/sheet/type/covid19_2021_by_location')


In [2]:
covid_dat = pd.read_csv( StringIO(response.text) ).iloc[1:,:-1].fillna(0)

In [3]:
covid_dat['datetime'] = pd.to_datetime( covid_dat['Ngày'] + '/2021' ,format= "%d/%m/%Y")

In [4]:
plot_data = covid_dat.drop(columns=['Ngày']).set_index('datetime').cumsum()
plot_dat = plot_data.stack().reset_index()
plot_dat['str_time'] = plot_dat['datetime'].apply(lambda x: int( x.strftime('%Y%m%d') ))

In [5]:
plot_dat
plot_json = plot_dat[plot_dat['datetime']<datetime.datetime.today()].drop(columns='datetime')
plot_json['level_1'] = plot_json['level_1'].apply(lambda val: no_accent_vietnamese(val))

plot_list = plot_json
plot_json = plot_json.values.tolist()

In [6]:
plot_json

[['Ha Noi', 0.0, 20210427],
 ['TP HCM', 0.0, 20210427],
 ['Hung Yen', 0.0, 20210427],
 ['Ha Nam', 0.0, 20210427],
 ['Vinh Phuc', 0.0, 20210427],
 ['Da Nang', 0.0, 20210427],
 ['Yen Bai', 1.0, 20210427],
 ['Quang Nam', 0.0, 20210427],
 ['Dong Nai', 0.0, 20210427],
 ['Hai Duong', 0.0, 20210427],
 ['Thai Binh', 0.0, 20210427],
 ['Quang Ngai', 0.0, 20210427],
 ['Lang Son', 0.0, 20210427],
 ['Bac Ninh', 0.0, 20210427],
 ['Thanh Hoa', 0.0, 20210427],
 ['Dien Bien', 0.0, 20210427],
 ['Nghe An', 0.0, 20210427],
 ['Nam Dinh', 0.0, 20210427],
 ['Phu Tho', 0.0, 20210427],
 ['Quang Ninh', 0.0, 20210427],
 ['Bac Giang', 0.0, 20210427],
 ['Hai Phong', 0.0, 20210427],
 ['Thua Thien Hue', 0.0, 20210427],
 ['Dak Lak', 0.0, 20210427],
 ['Hoa Binh', 0.0, 20210427],
 ['Quang Tri', 0.0, 20210427],
 ['Tuyen Quang', 0.0, 20210427],
 ['Son La', 0.0, 20210427],
 ['Ninh Binh', 0.0, 20210427],
 ['Thai Nguyen', 0.0, 20210427],
 ['Long An', 0.0, 20210427],
 ['Bac Lieu', 0.0, 20210427],
 ['Gia Lai', 0.0, 20210427],

In [7]:
json.dumps(list(plot_list['level_1'].unique()))

'["Ha Noi", "TP HCM", "Hung Yen", "Ha Nam", "Vinh Phuc", "Da Nang", "Yen Bai", "Quang Nam", "Dong Nai", "Hai Duong", "Thai Binh", "Quang Ngai", "Lang Son", "Bac Ninh", "Thanh Hoa", "Dien Bien", "Nghe An", "Nam Dinh", "Phu Tho", "Quang Ninh", "Bac Giang", "Hai Phong", "Thua Thien Hue", "Dak Lak", "Hoa Binh", "Quang Tri", "Tuyen Quang", "Son La", "Ninh Binh", "Thai Nguyen", "Long An", "Bac Lieu", "Gia Lai", "Tay Ninh", "Binh Duong", "Tra Vinh", "Dong Thap", "Ha Tinh", "Tien Giang", "Bac Kan", "Lao Cai", "An Giang", "Vinh Long", "Kien Giang", "Khanh Hoa", "Phu Yen", "Binh Thuan", "Can Tho", "Ba Ria - Vung Tau", "Binh Dinh", "Binh Phuoc", "Lam Dong", "Ninh Thuan", "Ben Tre", "Soc Trang", "Ca Mau", "Hau Giang", "Dak Nong"]'

In [8]:
final_plot_json = [['Country','Income','Year']] + plot_json

In [9]:
with open('covid_data.json', 'w') as the_file:
    json.dump(final_plot_json,the_file)